# 04 · ESM1b — a protein language model

ESM1b (Brandes et al. 2023, *Nat Genet*, PMID 37563329) is an *LLM for protein sequences*. Like EVE it is **unsupervised**, but it scores variants on a **log-likelihood ratio (LLR)** whose scale runs *backwards* from every other tool here.

> ⚠️ **DEMO DATA.** The ESM1b scores here are hand-authored illustrative values for ~13 curated CFTR variants — **not** real ESM1b output. Every table keeps a `source` column reading `DEMO`. See the *How to get REAL data* box, then join real per-variant scores on `protein_variant`; the code runs unchanged once `source` is `REAL`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 2 · ESM1b — a protein language model

**What is it?** ESM1b is a **protein language model** — think of it as an *LLM for protein sequences*. Just as a text LLM learns which word is likely to come next, ESM1b was trained on millions of natural protein sequences to predict which amino acid "belongs" at each position given its neighbours. It never saw the alignment explicitly; it learned protein grammar directly from raw sequences.

To score a variant, ESM1b compares how *likely* the model thinks the **mutant** amino acid is versus the **wild-type** amino acid at that position. This is a **log-likelihood ratio (LLR)**:

$$\text{LLR} = \log \frac{P(\text{mutant amino acid})}{P(\text{wild-type amino acid})}$$

> ### 🔄 IMPORTANT — ESM1b runs *backwards* from the other tools
> A **more negative** LLR means the model finds the mutation more *surprising* → **more damaging**. This is the **opposite direction** to EVE, AlphaMissense, and REVEL, where *higher* = worse.
> 
> - **Cut-off:** `LLR <= -7.5` ~ pathogenic. **Lower (more negative) = worse.**
> - Unsupervised (learned from sequences only) → low circularity, just like EVE.

> ### 📥 How to get REAL ESM1b data
> Bulk, per-protein ESM1b variant-effect files are published as the **Brandes et al. 2023 supplement** and on **HuggingFace**. Download the CFTR file and join on `protein_variant`.

In [2]:
# ESM1b demo scores. Again, source = DEMO. These are LLRs, so they are NEGATIVE.
esm = tk.load_esm1b()
print(f'{len(esm)} variants | columns: {list(esm.columns)}')
esm

13 variants | columns: ['protein_variant', 'esm1b_score', 'source']


,protein_variant,esm1b_score,source
0,G551D,-12.1,DEMO
2,R117H,-4.1,DEMO
3,R334W,-9.2,DEMO
4,G85E,-10.1,DEMO
5,D1152H,-5.4,DEMO
6,R668C,-3.1,DEMO
7,Tyr161Cys,-7.2,DEMO
8,Gly970Asp,-6.4,DEMO
9,Ser912Leu,-6.2,DEMO
10,Val520Phe,-6.0,DEMO


## 2 · Turn an ESM1b LLR into a call

`tk.call_from_score(score, 'esm1b')` bakes in the **direction**: cut at `<= -7.5`, and **lower / more negative = worse**. You do not juggle the sign yourself.

In [3]:
esm['esm1b_call'] = esm['esm1b_score'].apply(lambda s: tk.call_from_score(s, 'esm1b'))
esm[['protein_variant', 'esm1b_score', 'esm1b_call', 'source']]

,protein_variant,esm1b_score,esm1b_call,source
0,G551D,-12.1,pathogenic,DEMO
2,R117H,-4.1,benign,DEMO
3,R334W,-9.2,pathogenic,DEMO
4,G85E,-10.1,pathogenic,DEMO
5,D1152H,-5.4,benign,DEMO
6,R668C,-3.1,benign,DEMO
7,Tyr161Cys,-7.2,benign,DEMO
8,Gly970Asp,-6.4,benign,DEMO
9,Ser912Leu,-6.2,benign,DEMO
10,Val520Phe,-6.0,benign,DEMO


### The sign flip, made concrete

It is worth staring at *one* variant until the backwards scale clicks. Run the next cell.

In [4]:
# Most negative ESM1b = most damaging.
row = esm.sort_values('esm1b_score').iloc[0]
print(f"Variant {row['protein_variant']}:")
print(f"  ESM1b = {row['esm1b_score']:>6}  ->  {row['esm1b_call']:<10} "
      f"(LOW / negative score, cut at -7.5, so low = pathogenic)")
print('A more negative LLR = a more surprising mutation = more damaging —')
print('the opposite direction to EVE, AlphaMissense and REVEL.')

Variant G551D:
  ESM1b =  -12.1  ->  pathogenic (LOW / negative score, cut at -7.5, so low = pathogenic)
A more negative LLR = a more surprising mutation = more damaging —
the opposite direction to EVE, AlphaMissense and REVEL.


## Key takeaways

1. **ESM1b** scores the **log-likelihood ratio** of mutant vs wild-type amino acid. Cut `<= -7.5` ~ pathogenic — **lower / more negative = worse** (opposite to EVE).
2. `tk.call_from_score` handles the sign flip internally.
3. **Unsupervised** → low circularity, like EVE.
4. **DEMO** data — real per-protein ESM1b files ship with Brandes 2023 / HuggingFace; join on `protein_variant`.

**Next:** notebook 05 — **REVEL**, a *supervised* ensemble, and the circularity problem.